环境配置：
执行的环境安装指令
# 1. 创建环境
conda create -n tiny_vqa python=3.10 -y
conda activate tiny_vqa

# 2. 安装必要库 (Moondream 只需要 transformers 和 timm)
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
pip install transformers timm pillow pandas
pip install timm

trainer.py
用于模型、LoRA的构建、训练和验证

In [ ]:
import argparse
import os
import sys

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from PIL import Image
import pandas as pd
from safetensors.torch import load_file
from tqdm import tqdm

# 1. 路径与环境配置（确保无论从哪个工作目录运行脚本，也能正确找到模型与数据）
#    - SCRIPT_DIR: 当前脚本所在目录
#    - REPO_ROOT: 项目根目录（上一级目录）
#    - sys.path: 将项目根目录加入导入路径，方便导入 pretrain_model 包
SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
REPO_ROOT = os.path.abspath(os.path.join(SCRIPT_DIR, os.pardir))
sys.path.insert(0, REPO_ROOT)

from pretrain_model.moondream import MoondreamModel
from pretrain_model.config import MoondreamConfig
from transformers import AutoTokenizer


# 2. 构建原生 LoRA 权重模块 (手动实现矩阵分解)
#
# LoRA 原理：
#   对于每个线性层 W, 通过低秩分解 W + BA 来实现可训练的“增量”权重。
#   其中 A 的维度为 (r, in_features)，B 的维度为 (out_features, r)，r 为 LoRA rank。
#
# 本模块直接构造一个与原模型结构匹配的 LoRA 参数集合，并通过 get_lora_dict 生成给模型使用的结构。
class LoraModule(nn.Module):
    def __init__(self, n_layers, dim, ff_dim, qkv_dim, r=16):
        super().__init__()
        self.r = r
        self.n_layers = n_layers
        self.params = nn.ParameterDict()

        # 这里的参数命名要与预训练模型内部名称“对齐”，以便能够正确 injected 到 text_decoder 的 lora 参数结构中。
        for i in range(n_layers):
            # Attention 层的 LoRA 参数：qkv 和 proj 分别对应 self-attention 的两部分线性变换
            self.params[f"layer_{i}_attn_qkv_A"] = nn.Parameter(torch.randn(r, dim) * 0.01)
            self.params[f"layer_{i}_attn_qkv_B"] = nn.Parameter(torch.zeros(qkv_dim, r))
            self.params[f"layer_{i}_attn_proj_A"] = nn.Parameter(torch.randn(r, dim) * 0.01)
            self.params[f"layer_{i}_attn_proj_B"] = nn.Parameter(torch.zeros(dim, r))

            # MLP 层的 LoRA 参数：对应 transformer block 里的前向网络 fc1 / fc2
            self.params[f"layer_{i}_mlp_fc1_A"] = nn.Parameter(torch.randn(r, dim) * 0.01)
            self.params[f"layer_{i}_mlp_fc1_B"] = nn.Parameter(torch.zeros(ff_dim, r))
            self.params[f"layer_{i}_mlp_fc2_A"] = nn.Parameter(torch.randn(r, ff_dim) * 0.01)
            self.params[f"layer_{i}_mlp_fc2_B"] = nn.Parameter(torch.zeros(dim, r))

    def get_lora_dict(self):
        """构造符合源码 text_decoder 期待的递归字典结构。

        Moondream 的 text_decoder 接收一个 `lora` 参数，它是一个多层嵌套的 dict：
          lora["text"]["blocks"][i]["attn"]["qkv"]["A"] 等

        该函数将当前 Module 中的 Parameter 组织成这种结构，以便训练 / 推理时直接传给 text_decoder。
        """
        lora_structure = {"text": {"blocks": {}}}
        for i in range(self.n_layers):
            lora_structure["text"]["blocks"][str(i)] = {
                "attn": {
                    "qkv": {"A": self.params[f"layer_{i}_attn_qkv_A"], "B": self.params[f"layer_{i}_attn_qkv_B"]},
                    "proj": {"A": self.params[f"layer_{i}_attn_proj_A"], "B": self.params[f"layer_{i}_attn_proj_B"]},
                },
                "mlp": {
                    "fc1": {"A": self.params[f"layer_{i}_mlp_fc1_A"], "B": self.params[f"layer_{i}_mlp_fc1_B"]},
                    "fc2": {"A": self.params[f"layer_{i}_mlp_fc2_A"], "B": self.params[f"layer_{i}_mlp_fc2_B"]},
                },
            }
        return lora_structure


# 3. 针对医学 VQA 的自定义数据集 (保持 PIL 格式以适配模型编码器)
#
# 该 Dataset 读取 CSV 并按“行”返回一个训练样本：
#   - image: PIL Image (延迟到模型内部做预处理)
#   - prompt: 文本 prompt，用于与图像特征一起输入 decoder
#   - target_id: 目标标签的 token id（用于计算交叉熵 loss）
class MedicalVQADataset(Dataset):
    def __init__(
        self,
        csv_path,
        img_dir,
        tokenizer,
        max_samples=None,
        shuffle=True,
    ):
        # 读取 CSV 并选择子集，方便快速实验与调参
        df = pd.read_csv(csv_path)
        if shuffle:
            # 随机打乱，有助于训练稳定
            df = df.sample(frac=1, random_state=42).reset_index(drop=True)
        if max_samples is not None:
            # 只取前 N 条，加速调试
            df = df.head(max_samples)
        self.data = df
        self.img_dir = img_dir
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # 逐条读取数据，构建模型所需输入
        row = self.data.iloc[idx]
        img_path = os.path.join(self.img_dir, row["Figure_path"])

        # 1) 图像：保持 PIL 格式，模型内部负责预处理 + 特征提取
        image = Image.open(img_path).convert("RGB")

        # 2) 目标标签：答案通常是单个字母 (A/B/C/D)
        target_char = str(row["Answer"]).strip().upper()
        target_ids = self.tokenizer.encode(target_char, add_special_tokens=False)
        target_id = target_ids[0] if len(target_ids) > 0 else self.tokenizer.unk_token_id

        # 3) Prompt 构造：插入问题和选项信息，末尾保留 Answer: 让模型生成答案
        prompt = (
            f"Question: {row['Question']}\n"
            f"Choices: A: {row['Choice A']}, B: {row['Choice B']}, C: {row['Choice C']}, D: {row['Choice D']}\n"
            "Answer:"
        )

        return image, prompt, target_id


def build_mask(seq_len: int, visual_token_count: int = 730, device=None):
    """构建 Moondream 的混合 Attention Mask。

    Moondream 结构：
      - 前 730 个 token（BOS + 图像 patch）使用全向注意力（bidirectional attention），以便图像特征之间互相有信息流动
      - 后续文本 token 使用因果自回归注意力（causal attention），保证 decoder 生成时只能看到前面的 token

    Args:
        seq_len: 当前输入 token 数（BOS + 图像 + prompt）
        visual_token_count: 需要全向注意力的前置 token 数，默认为 730
        device: mask 所在的设备
    """
    mask = torch.zeros(seq_len, seq_len, device=device, dtype=torch.bool)
    mask[:visual_token_count, :visual_token_count] = True
    for j in range(visual_token_count, seq_len):
        mask[j, : j + 1] = True
    return mask

 
def evaluate(
    model,
    lora_dict,
    tokenizer,
    config,
    dataset,
    device,
    max_examples=None,
    verbose=False,
):
    """在给定数据集上评估 LoRA 适配器的准确率和平均损失。

    说明：
      - 该函数用于验证集/开发集评估，可在每个 epoch 结束后调用
      - 由于训练过程中使用了 LoRA，在 eval 时同样需要把 LoRA 注入到 text_decoder 中
    """
    from pretrain_model.text import text_encoder, text_decoder, lm_head

    model.eval()
    correct = 0
    total = 0
    total_loss = 0.0

    with torch.no_grad():
        for idx in range(len(dataset)):
            if max_examples is not None and total >= max_examples:
                break

            # 1）取数据
            image, prompt, target_id = dataset[idx]

            # 2）图像编码：注意保持与训练一致的处理方式
            img_emb = model._run_vision_encoder(image)
            if img_emb.dim() == 2:
                img_emb = img_emb.unsqueeze(0)

            # 3）文本编码（Prompt）
            prompt_ids = tokenizer.encode(prompt, add_special_tokens=False)
            prompt_toks = torch.tensor([prompt_ids], device=device)
            prompt_emb = text_encoder(prompt_toks, model.text)

            # 4）BOS + 图像 + Prompt 拼接
            bos_id = torch.tensor([[config.tokenizer.bos_id]], device=device)
            bos_emb = text_encoder(bos_id, model.text)
            inputs_embeds = torch.cat([bos_emb, img_emb, prompt_emb], dim=1)

            # 5）构造 attention mask（与训练一致）
            seq_len = inputs_embeds.size(1)
            mask = build_mask(seq_len, device=device)
            pos_ids = torch.arange(seq_len, device=device)

            # 6）前向计算（注入 LoRA）
            hidden = text_decoder(
                inputs_embeds,
                model.text,
                mask,
                pos_ids,
                config.text,
                lora=lora_dict,
            )

            logits = lm_head(hidden, model.text)

            # 7）损失/准确率计算（只比较最后一个 token）
            loss = F.cross_entropy(logits, torch.tensor([target_id], device=device))
            total_loss += loss.item()

            # --- 引入约束解码 (Constrained Decoding) ---
            valid_chars = ['A', 'B', 'C', 'D']
            # 动态获取当前 tokenizer 中 ABCD 的 token_id
            valid_ids = [tokenizer.encode(c, add_special_tokens=False)[0] for c in valid_chars]
            
            # 只在 A/B/C/D 四个维度中取局部最大值
            target_logits = logits[0, valid_ids]
            best_local_idx = torch.argmax(target_logits).item()
            pred_id = valid_ids[best_local_idx]
            
            if pred_id == target_id:
                correct += 1
            total += 1

            if verbose and total % 50 == 0:
                print(f"  eval {total}/{len(dataset)}  acc={(correct/total):.3f}")

    return {
        "accuracy": correct / max(total, 1),
        "loss": total_loss / max(total, 1),
        "samples": total,
    }


def train(args):
    # 训练入口，支持命令行参数传递（见 parse_args）
    # device: 自动选择 CUDA/CPU
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # 1) 模型初始化（基座模型 + LoRA 模块）
    config = MoondreamConfig()
    model = MoondreamModel(config).to(device)

    print("[1/5] 加载基座模型权重...")
    weights_path = os.path.join(SCRIPT_DIR, "pretrain_model", "model.safetensors")
    state_dict = load_file(weights_path)
    model.load_state_dict(state_dict, strict=False)

    # 2) 冻结基座模型参数，只训练 LoRA 参数（降低显存与计算）
    for param in model.parameters():
        param.requires_grad = False

    # 3) 关闭 KV Cache：避免在训练过程中因缓存状态导致行为差异
    for block in model.text.blocks:
        block.kv_cache = None

    # 4) 构造 LoRA 模块并设置优化器（只更新 LoRA 参数）
    qkv_dim = int(config.text.dim * (1 + 2 * config.text.n_kv_heads / config.text.n_heads))
    lora_module = LoraModule(
        config.text.n_layers, config.text.dim, config.text.ff_dim, qkv_dim
    ).to(device=device, dtype=torch.bfloat16)

    optimizer = torch.optim.AdamW(lora_module.parameters(), lr=args.lr)

    # 5) Tokenizer 加载：与基座模型保持一致
    tokenizer = AutoTokenizer.from_pretrained(
        os.path.join(SCRIPT_DIR, "pretrain_model"), local_files_only=True
    )

    # 6) 构建训练/验证集
    train_ds = MedicalVQADataset(
        os.path.join(REPO_ROOT, args.train_csv),
        os.path.join(REPO_ROOT, args.img_dir),
        tokenizer,
        max_samples=args.max_train_samples,
        shuffle=True,
    )
    val_ds = None
    if args.val_csv:
        val_ds = MedicalVQADataset(
            os.path.join(REPO_ROOT, args.val_csv),
            os.path.join(REPO_ROOT, args.img_dir),
            tokenizer,
            max_samples=args.max_val_samples,
            shuffle=False,
        )

    print(f"[2/5] 训练集 {len(train_ds)} 样本", end="")
    if val_ds is not None:
        print(f"，验证集 {len(val_ds)} 样本")
    else:
        print("")

    best_acc = 0.0
    history = []

    from pretrain_model.text import text_encoder, text_decoder, lm_head

    print("[3/5] 开始训练 Loop...")
    for epoch in range(1, args.epochs + 1):
        model.train()
        total_loss = 0.0

        # 改进进度条：显示当前 loss、平均 loss、lr，并用更直观的 ‘step’ 单位。
        pbar = tqdm(
            train_ds,
            desc=f"Epoch {epoch}/{args.epochs}",
            unit="step",
            mininterval=0.5,
            leave=False,
        )

        # 训练循环：每个 Step 处理一个样本（可扩展为 batch）
        for i, (image, prompt, target_id) in enumerate(pbar, start=1):
            # 1) 梯度清零 & 准备计算
            optimizer.zero_grad()

            # 2) 视觉编码（不计算梯度，节省显存）
            #    该步骤仅提取基础的视觉特征，后续与文本一起输入 decoder。
            with torch.no_grad():
                img_emb = model._run_vision_encoder(image)
                if img_emb.dim() == 2:
                    # 视觉特征有时返回 [seq, dim]，需要增加 batch 维度
                    img_emb = img_emb.unsqueeze(0)

            # 3) 文本编码：将 prompt 转为 embedding
            prompt_ids = tokenizer.encode(prompt, add_special_tokens=False)
            prompt_toks = torch.tensor([prompt_ids], device=device)
            prompt_emb = text_encoder(prompt_toks, model.text)

            # 4) 构造输入序列：BOS + 图像特征 + prompt 文本特征
            bos_id = torch.tensor([[config.tokenizer.bos_id]], device=device)
            bos_emb = text_encoder(bos_id, model.text)
            inputs_embeds = torch.cat([bos_emb, img_emb, prompt_emb], dim=1)

            # 5) 构造 Attention mask 与 position ids（与训练 / 推理保持一致）
            seq_len = inputs_embeds.size(1)
            mask = build_mask(seq_len, device=device)
            pos_ids = torch.arange(seq_len, device=device)

            # 6) 前向传播：将 LoRA 注入到 text_decoder
            hidden = text_decoder(
                inputs_embeds,
                model.text,
                mask,
                pos_ids,
                config.text,
                lora=lora_module.get_lora_dict(),
            )

            # 7) 输出 logits + 计算 loss（只取最后一个 token）
            logits = lm_head(hidden, model.text)
            loss = F.cross_entropy(logits, torch.tensor([target_id], device=device))
            loss.backward()
            optimizer.step()

            # 8) 统计并展示训练进度
            total_loss += loss.item()
            avg_loss = total_loss / i
            pbar.set_postfix(
                {
                    "loss": f"{loss.item():.4f}",
                    "avg": f"{avg_loss:.4f}",
                    "lr": f"{optimizer.param_groups[0]['lr']:.2e}",
                }
            )

        train_loss = total_loss / len(train_ds)

        eval_stats = None
        if val_ds is not None:
            eval_stats = evaluate(
                model,
                lora_module.get_lora_dict(),
                tokenizer,
                config,
                val_ds,
                device,
                max_examples=args.max_val_samples,
                verbose=False,
            )

        epoch_info = {
            "epoch": epoch,
            "train_loss": train_loss,
            **({"val_loss": eval_stats["loss"], "val_acc": eval_stats["accuracy"]} if eval_stats else {}),
        }
        history.append(epoch_info)

        print(
            f"Epoch {epoch}/{args.epochs} | train_loss={train_loss:.4f} "
            + (f"| val_loss={eval_stats['loss']:.4f} val_acc={eval_stats['accuracy']:.3f}" if eval_stats else "")
        )

        # 每个 epoch 结束后，如果当前模型在验证集上表现更好，则保存一份“最优” LoRA 权重。
        # 这样可避免最后一个 epoch 不一定是最优模型的情况。
        if eval_stats and eval_stats["accuracy"] > best_acc:
            best_acc = eval_stats["accuracy"]
            save_path = os.path.join(args.save_dir, f"best_lora_epoch{epoch}.pt")
            os.makedirs(args.save_dir, exist_ok=True)
            torch.save(lora_module.state_dict(), save_path)
            print(f"  ✅ 保存最佳 LoRA 适配器: {save_path} (val_acc={best_acc:.3f})")

    # 最终保存：无论是否提升，都会将当前 LoRA 权重保存为通用文件名
    final_path = os.path.join(args.save_dir, "medical_lora_adapter.pt")
    os.makedirs(args.save_dir, exist_ok=True)
    torch.save(lora_module.state_dict(), final_path)
    print(f"训练结束，LoRA 权重已保存: {final_path}")

    # 输出训练、验证历史，便于后续分析与记录
    print("训练历史:")
    for row in history:
        print(row)


def parse_args():
    parser = argparse.ArgumentParser(description="Moondream Medical VQA LoRA 训练脚本")
    parser.add_argument("--train-csv", type=str, default="PMC_VQA/test_2.csv", help="训练集 CSV 路径")
    parser.add_argument("--val-csv", type=str, default="PMC_VQA/test_2.csv", help="验证集 CSV 路径（可与训练集相同，或置空不做验证）")
    parser.add_argument("--img-dir", type=str, default="PMC_VQA/images_2/figures", help="图片目录")
    parser.add_argument("--epochs", type=int, default=1, help="训练轮数")
    parser.add_argument("--lr", type=float, default=5e-5, help="学习率")
    parser.add_argument("--max-train-samples", type=int, default=2000, help="最多训练样本数")
    parser.add_argument("--max-val-samples", type=int, default=2000, help="最多验证样本数")
    parser.add_argument("--save-dir", type=str, default=os.path.join(SCRIPT_DIR, "result"), help="LoRA 权重保存目录")
    return parser.parse_args()


if __name__ == "__main__":
    args = parse_args()
    train(args)


终端命令：
python .\test\trainer.py --epochs 2 --lr 5e-5 --max-train-samples 200 --max-val-samples 50 --val-csv "PMC_VQA/test_2.csv"

test.py
用于模型、LoRA的构建、训练和验证

In [ ]:
import argparse
import os
import sys

import torch
import pandas as pd
from PIL import Image
from safetensors.torch import load_file

# ============================
# 评估脚本说明：
# - 用于加载训练好的 LoRA 适配器（medical_lora_adapter.pt），并在 PMC_VQA 测试集上做推理
# - 只输出 A/B/C/D 四个选项中的一个（与训练时的任务一致）
# - 结果以 CSV 形式保存，可用于进一步分析（例如准确率、混淆矩阵等）
#
# 使用示例：
#   python test/test.py --csv PMC_VQA/test_2.csv --img-dir PMC_VQA/images_2/figures --lora-path result/medical_lora_adapter.pt
# ============================

# 1. 路径与环境配置（确保无论从哪个目录运行此脚本，都能找到模型与数据）
#
# - SCRIPT_DIR: 当前脚本所在目录
# - REPO_ROOT: 项目根目录（作为相对路径计算的基准）
# - sys.path.insert: 将 REPO_ROOT 加入导入路径，便于直接 import pretrain_model
SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
REPO_ROOT = os.path.abspath(os.path.join(SCRIPT_DIR, os.pardir))
sys.path.insert(0, REPO_ROOT)

from pretrain_model.moondream import MoondreamModel
from pretrain_model.config import MoondreamConfig
from transformers import AutoTokenizer


class LoraModule(torch.nn.Module):
    """LoRA adapter 参数容器（与 trainer.py 保持一致）。

    本脚本只用来加载/保存 LoRA 适配器权重，因此结构需与训练时一致。

    结构说明：
      - 每个 transformer layer 里，LoRA 修改了 attention 的 qkv/proj 线性层与 mlp 的两个线性层
      - 这里采用 A/B 两个矩阵表示低秩分解：W_delta ≈ B @ A
      - 最终通过 get_lora_dict() 生成符合 pretrain_model.text_decoder 期待格式的字典
    """

    def __init__(self, n_layers, dim, ff_dim, qkv_dim, r=16):
        super().__init__()
        self.r = r
        self.n_layers = n_layers
        # 使用 ParameterDict 方便统一保存/加载权重，同时保持与 torch.nn.Module 兼容
        self.params = torch.nn.ParameterDict()

        # 初始化 LoRA 权重为小的随机值，以便后续训练稳定收敛
        for i in range(n_layers):
            self.params[f"layer_{i}_attn_qkv_A"] = torch.nn.Parameter(torch.randn(r, dim) * 0.01)
            self.params[f"layer_{i}_attn_qkv_B"] = torch.nn.Parameter(torch.zeros(qkv_dim, r))
            self.params[f"layer_{i}_attn_proj_A"] = torch.nn.Parameter(torch.randn(r, dim) * 0.01)
            self.params[f"layer_{i}_attn_proj_B"] = torch.nn.Parameter(torch.zeros(dim, r))
            self.params[f"layer_{i}_mlp_fc1_A"] = torch.nn.Parameter(torch.randn(r, dim) * 0.01)
            self.params[f"layer_{i}_mlp_fc1_B"] = torch.nn.Parameter(torch.zeros(ff_dim, r))
            self.params[f"layer_{i}_mlp_fc2_A"] = torch.nn.Parameter(torch.randn(r, ff_dim) * 0.01)
            self.params[f"layer_{i}_mlp_fc2_B"] = torch.nn.Parameter(torch.zeros(dim, r))

    def get_lora_dict(self):
        """将内部 ParameterDict 转换为 TextDecoder 所需的 LoRA 字典结构。

        返回的 dict 结构如下：
          {
            "text": {
              "blocks": {
                "0": {"attn": {...}, "mlp": {...}},
                "1": {...},
                ...
              }
            }
          }
        """
        lora_structure = {"text": {"blocks": {}}}
        for i in range(self.n_layers):
            lora_structure["text"]["blocks"][str(i)] = {
                "attn": {
                    "qkv": {"A": self.params[f"layer_{i}_attn_qkv_A"], "B": self.params[f"layer_{i}_attn_qkv_B"]},
                    "proj": {"A": self.params[f"layer_{i}_attn_proj_A"], "B": self.params[f"layer_{i}_attn_proj_B"]},
                },
                "mlp": {
                    "fc1": {"A": self.params[f"layer_{i}_mlp_fc1_A"], "B": self.params[f"layer_{i}_mlp_fc1_B"]},
                    "fc2": {"A": self.params[f"layer_{i}_mlp_fc2_A"], "B": self.params[f"layer_{i}_mlp_fc2_B"]},
                },
            }
        return lora_structure


def load_model_and_lora(args, device):
    """加载基座 Moondream 模型 + LoRA 适配器权重，并返回推理所需的各类对象。

    Args:
        args: 命令行参数（包含 lora_path）
        device: 运行设备（'cuda' 或 'cpu'）

    Returns:
        model: 基座 MoondreamModel（已加载权重、设置为 eval 模式）
        lora_dict: 适用于 pretrain_model.text_decoder 的 LoRA 字典结构
        tokenizer: 用于将文本转换为 token id 的 tokenizer
        config: MoondreamConfig 实例
    """

    # 1) 加载模型配置和基座模型（权重尚未加载）
    config = MoondreamConfig()
    model = MoondreamModel(config).to(device)

    # 2) 加载预训练的基座权重
    # strict=False 允许模型结构与权重文件之间存在不完全匹配（例如新增/缺失键），
    # 因为我们只关心加载大部分的模型权重，用于 inference。
    weights_path = os.path.join(SCRIPT_DIR, "pretrain_model", "model.safetensors")
    model.load_state_dict(load_file(weights_path), strict=False)

    # 3) 清理推理时可能残留的缓存状态（如 kv_cache），保证评估过程不受之前运行影响。
    for block in model.text.blocks:
        block.kv_cache = None
    model.eval()

    # 4) 构建与训练中一致的 LoRA 模块结构，并加载权重
    #    qkv_dim 计算方式需与模型内部的 qkv 映射一致
    qkv_dim = int(config.text.dim * (1 + 2 * config.text.n_kv_heads / config.text.n_heads))
    lora_module = LoraModule(config.text.n_layers, config.text.dim, config.text.ff_dim, qkv_dim).to(
        device=device, dtype=torch.bfloat16
    )

    lora_ckpt = args.lora_path or os.path.join(SCRIPT_DIR, "result", "medical_lora_adapter.pt")
    lora_module.load_state_dict(torch.load(lora_ckpt, map_location=device))

    # 5) 加载 tokenizer（用于将 prompt 从文本转为 token id）
    tokenizer = AutoTokenizer.from_pretrained(os.path.join(SCRIPT_DIR, "pretrain_model"), local_files_only=True)

    return model, lora_module.get_lora_dict(), tokenizer, config


def predict_one(model, lora_dict, tokenizer, config, device, image, prompt):
    """对单条样本进行推理，输出预测的选项（A/B/C/D）。

    过程说明：
      1) 图像先经过视觉编码器得到图像嵌入（img_emb）
      2) 文本 prompt（包含问题 + 选项）被 tokenizer 编码并经过 text_encoder 得到文本嵌入
      3) 将 BOS token 嵌入 + 图像嵌入 + 文本嵌入拼接到一起作为 decoder 的输入
      4) 构造 causal attention mask（前 730 token 允许双向注意力，之后为因果注意力）
      5) 调用 text_decoder 得到隐层表示，再用 lm_head 输出 logits
      6) 从 logits 中挑选对应选项 A/B/C/D 的 token logits，并选出最大值对应的选项
    """
    # 仅在此函数内导入模块，避免在不需要时诱发循环依赖
    from pretrain_model.text import text_encoder, text_decoder, lm_head

    # 1) 图像 -> 图像嵌入（batch=1）
    img_emb = model._run_vision_encoder(image)
    if img_emb.dim() == 2:
        # 视觉编码器可能返回 [seq_len, dim]，这里统一变成 [1, seq_len, dim]
        img_emb = img_emb.unsqueeze(0)

    # 2) 文本 prompt -> token ids -> 文本嵌入
    prompt_ids = tokenizer.encode(prompt, add_special_tokens=False)
    prompt_toks = torch.tensor([prompt_ids], device=device)
    prompt_emb = text_encoder(prompt_toks, model.text)

    # 3) 在 decoder 输入序列前加上 BOS token 的嵌入（与训练时输入保持一致）
    bos_id = torch.tensor([[config.tokenizer.bos_id]], device=device)
    bos_emb = text_encoder(bos_id, model.text)
    inputs_embeds = torch.cat([bos_emb, img_emb, prompt_emb], dim=1)

    # 4) 构造 attention mask：
    #    - 前 730 个 token 允许全局双向 attention（用于图像 + prompt 的 cross-attention）
    #    - 之后 token 采用因果注意力（生成式推理只看前文）
    seq_len = inputs_embeds.size(1)
    mask = torch.zeros(seq_len, seq_len, device=device, dtype=torch.bool)
    mask[:730, :730] = True
    for j in range(730, seq_len):
        mask[j, : j + 1] = True

    pos_ids = torch.arange(seq_len, device=device)

    # 5) 运行 decoder，并得出每个位置的 logits
    hidden = text_decoder(inputs_embeds, model.text, mask, pos_ids, config.text, lora=lora_dict)
    logits = lm_head(hidden, model.text)

    # 6) 仅比较 A/B/C/D 4 个 token 的 logits，然后选最大值对应的预测字符
    valid_chars = ["A", "B", "C", "D"]
    valid_ids = [tokenizer.encode(c, add_special_tokens=False)[0] for c in valid_chars]
    target_logits = logits[0, valid_ids]
    best_idx = torch.argmax(target_logits).item()
    return valid_chars[best_idx]


def run_eval(args):
    """主评估入口。

    该函数负责：
      1) 初始化模型和 tokenizer
      2) 读取测试集 CSV、逐行遍历样本
      3) 调用 predict_one 得到预测结果
      4) 统计准确率，并将每条预测结果写入输出 CSV
    """

    # 选择运行设备（优先使用 GPU）
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # 加载模型和 LoRA 权重
    model, lora_dict, tokenizer, config = load_model_and_lora(args, device)

    # 读取测试数据
    df = pd.read_csv(os.path.join(REPO_ROOT, args.csv))
    if args.max_samples is not None:
        df = df.head(args.max_samples)

    img_dir = os.path.join(REPO_ROOT, args.img_dir)

    results = []
    correct = 0

    # 遍历每条样本并推理
    for idx, row in df.iterrows():
        image_path = os.path.join(img_dir, row["Figure_path"])
        if not os.path.exists(image_path):
            # 如果图片文件缺失，记录错误并跳过
            results.append({"Figure_path": row["Figure_path"], "error": "IMAGE_NOT_FOUND"})
            continue

        image = Image.open(image_path).convert("RGB")

        # 构造 prompt：训练时使用同样的格式
        prompt = (
            f"Question: {row['Question']}\n"
            f"Choices: A: {row['Choice A']}, B: {row['Choice B']}, C: {row['Choice C']}, D: {row['Choice D']}\n"
            "Answer:"
        )

        # 预测并统计准确率
        pred = predict_one(model, lora_dict, tokenizer, config, device, image, prompt)
        actual = str(row.get("Answer", "")).strip().upper()
        is_correct = (pred == actual)
        correct += 1 if is_correct else 0

        results.append(
            {
                "Figure_path": row["Figure_path"],
                "Question": row["Question"],
                "Actual": actual,
                "Predicted": pred,
                "Is_Correct": is_correct,
            }
        )

        # 可选的中间结果输出（方便观察评估进度）
        if args.verbose and (idx + 1) % 50 == 0:
            acc = correct / (idx + 1)
            print(f"[{idx+1}/{len(df)}] Acc={acc:.3%}")

    # 将评估结果写入 CSV（用于后续分析）
    output_df = pd.DataFrame(results)
    output_csv = args.output_csv or os.path.join(REPO_ROOT, "pmc_vqa_results.csv")
    output_df.to_csv(output_csv, index=False)

    acc = correct / len(df) if len(df) > 0 else 0.0
    print(f"完成评估: 总样本={len(df)}, 准确率={acc:.3%}, 结果已保存至 {output_csv}")


def parse_args():
    """命令行参数解析函数。

    运行示例：
      python test/test.py --csv PMC_VQA/test_2.csv --img-dir PMC_VQA/images_2/figures --lora-path result/medical_lora_adapter.pt
    """
    parser = argparse.ArgumentParser(description="Moondream Medical VQA 评估脚本")
    parser.add_argument("--csv", type=str, default="PMC_VQA/test_2.csv", help="待评估 CSV 路径")
    parser.add_argument("--img-dir", type=str, default="PMC_VQA/images_2/figures", help="图片目录")
    parser.add_argument("--lora-path", type=str, default=None, help="LoRA 权重文件路径")
    parser.add_argument("--max-samples", type=int, default=200, help="最多评估样本数")
    parser.add_argument("--output-csv", type=str, default=None, help="输出结果 CSV 路径")
    parser.add_argument("--verbose", action="store_true", help="打印评估过程中的中间准确率")
    return parser.parse_args()


if __name__ == "__main__":
    # 当脚本作为主程序运行时，解析命令行参数并执行评估
    run_eval(parse_args())


python test/test.py    